# 09 — Comparator specificity analysis

## Objective

This revision notebook evaluates whether the vitamin D transcriptional core and selected recurrent Hallmark signals are quantitatively enriched in vitamin D-related perturbations rather than merely reflecting generic LINCS perturbational responses.

## Comparator strategy

The fixed vitamin D core is projected onto three LINCS Level 5 signature groups:

1. `vitamin_D_related`: manuscript-defining vitamin D-related signatures.
2. `nuclear_receptor_comparator`: non-VDR nuclear receptor-related perturbations.
3. `unrelated_comparator`: eligible compound perturbations without vitamin D, VDR, or nuclear receptor annotation.

The vitamin D core is not re-derived in comparator signatures. This avoids circularity and directly tests enrichment of the manuscript-defined signal in comparator backgrounds.

## Main controls

- Same manuscript cell lines: A549, HA1E, MCF7, PC3, U2OS.
- 24 h compound perturbations only.
- Replicate-supported signatures (`nsample >= 3`).
- Cell-line-matched resampling for both fixed-core and pathway-level comparisons.

## Main output

`results/revision/tables/comparator_specificity_summary.csv`


In [ ]:
from pathlib import Path
import pickle
import sys
import warnings

import h5py
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

RANDOM_SEED = 2026
TARGET_CELLS = ["A549", "HA1E", "MCF7", "PC3", "U2OS"]

N_CORE_RESAMPLING_ITERATIONS = 5000
N_PATHWAY_RESAMPLING_ITERATIONS = 5000
BLOCK_SIZE = 512
NUMERICAL_TOLERANCE = 1e-5


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "data" / "raw_data").exists()
            and (candidate / "data" / "exports").exists()
            and (candidate / "libs").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate project root. Expected folders: data/raw_data, data/exports, and libs."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
EXPORTS_DIR = DATA_DIR / "exports"
LIBS_DIR = PROJECT_ROOT / "libs"

RESULTS_DIR = PROJECT_ROOT / "results" / "revision"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "level5": RAW_DIR / "level5_beta_trt_cp_n720216x12328.gctx",
    "siginfo": RAW_DIR / "siginfo_beta.txt",
    "compoundinfo": RAW_DIR / "compoundinfo_beta.txt",
    "geneinfo": RAW_DIR / "geneinfo_beta.txt",
}

EXPORT_FILES = {
    "vitd_metadata_core": EXPORTS_DIR / "signature_metadata_with_core_score.csv",
    "vitd_signatures": EXPORTS_DIR / "subset_signatures_meta.csv",
}

CANONICAL_CORE_PATH = PROJECT_ROOT / "results" / "sensitivity" / "historical" / "notebook_root" / "core_v2.pkl"
CORE_SCORES_TOP50_PATH = PROJECT_ROOT / "results" / "sensitivity" / "historical" / "notebook_root" / "core_scores_top50.csv"
HALLMARK_GMT_PATH = LIBS_DIR / "h.all.v2025.1.Hs.symbols.gmt"

REQUIRED_FILES = {
    **RAW_FILES,
    **EXPORT_FILES,
    "canonical_core": CANONICAL_CORE_PATH,
    "canonical_core_scores_top50": CORE_SCORES_TOP50_PATH,
    "hallmark_gmt": HALLMARK_GMT_PATH,
}

missing_files = [
    f"{name}: {path.relative_to(PROJECT_ROOT)}"
    for name, path in REQUIRED_FILES.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing_files))

SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from vitd_utils import coregenes

print("Project root")
print("-" * 80)
print(PROJECT_ROOT)

print("\nKey input files")
print("-" * 80)
for name, path in REQUIRED_FILES.items():
    print(f"{name:30s} {path.relative_to(PROJECT_ROOT)}")


## 1. Load metadata and audit identifiers

This section loads LINCS metadata and verifies that manuscript-defining vitamin D signatures are present in both `siginfo` and the Level 5 GCTX matrix.


In [ ]:
SIGINFO_COLUMNS = [
    "sig_id", "pert_id", "pert_type", "cell_iname", "pert_time", "pert_time_unit",
    "pert_dose", "pert_dose_unit", "nearest_dose", "pert_idose", "nsample",
    "cc_q75", "ss_ngene", "tas", "qc_pass", "is_hiq", "is_exemplar_sig",
    "is_ncs_sig", "is_null_sig", "cmap_name",
]

siginfo = pd.read_csv(RAW_FILES["siginfo"], sep="\t", usecols=SIGINFO_COLUMNS, low_memory=False)
compoundinfo = pd.read_csv(RAW_FILES["compoundinfo"], sep="\t", low_memory=False)
geneinfo = pd.read_csv(RAW_FILES["geneinfo"], sep="\t", low_memory=False)

vitd_meta_core = pd.read_csv(EXPORT_FILES["vitd_metadata_core"])
vitd_signatures = pd.read_csv(EXPORT_FILES["vitd_signatures"])


def decode_hdf5_values(values):
    return [x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in values]


with h5py.File(RAW_FILES["level5"], "r") as h5:
    gctx_col_ids = decode_hdf5_values(h5["0/META/COL/id"][:])
    gctx_row_ids = decode_hdf5_values(h5["0/META/ROW/id"][:])
    gctx_matrix_shape = h5["0/DATA/0/matrix"].shape

gctx_col_id_set = set(gctx_col_ids)
gctx_row_id_set = set(gctx_row_ids)

siginfo["sig_id"] = siginfo["sig_id"].astype(str)
siginfo["pert_id"] = siginfo["pert_id"].astype(str)

vitd_core_sig_ids = set(vitd_meta_core["sig_id"].astype(str))
vitd_all_sig_ids = set(vitd_signatures["sig_id"].astype(str))
siginfo_sig_ids = set(siginfo["sig_id"])

expected_shape = (len(gctx_col_ids), len(gctx_row_ids))

print("Loaded metadata")
print("-" * 80)
print(f"siginfo:          {siginfo.shape}")
print(f"compoundinfo:     {compoundinfo.shape}")
print(f"geneinfo:         {geneinfo.shape}")
print(f"vitd_meta_core:   {vitd_meta_core.shape}")
print(f"vitd_signatures:  {vitd_signatures.shape}")

print("\nGCTX audit")
print("-" * 80)
print(f"GCTX matrix shape:              {gctx_matrix_shape}")
print(f"GCTX column/signature IDs:      {len(gctx_col_ids)}")
print(f"GCTX row/gene IDs:              {len(gctx_row_ids)}")
print(f"vitD core sig_ids in siginfo:   {len(vitd_core_sig_ids & siginfo_sig_ids)} / {len(vitd_core_sig_ids)}")
print(f"vitD core sig_ids in GCTX:      {len(vitd_core_sig_ids & gctx_col_id_set)} / {len(vitd_core_sig_ids)}")

if gctx_matrix_shape != expected_shape:
    raise ValueError(f"Unexpected GCTX matrix shape: {gctx_matrix_shape}; expected {expected_shape}")

if len(vitd_core_sig_ids & siginfo_sig_ids) != len(vitd_core_sig_ids):
    raise ValueError("Not all manuscript vitamin D signatures are present in siginfo.")

if len(vitd_core_sig_ids & gctx_col_id_set) != len(vitd_core_sig_ids):
    raise ValueError("Not all manuscript vitamin D signatures are present in the GCTX matrix.")

print("\nIdentifier audit passed.")


## 2. Define and classify the comparator universe

The comparator universe mirrors the original manuscript filters where possible: Level 5 signatures, compound perturbations, 24 h treatment, the same five cell lines, and `nsample >= 3`.

`qc_pass` is not used as an eligibility filter because the manuscript-defining vitamin D set includes both `qc_pass == 1` and `qc_pass == 0` signatures. Applying a stricter QC rule only to comparators would make the comparison asymmetric.


In [ ]:
siginfo_gctx = siginfo[siginfo["sig_id"].isin(gctx_col_id_set)].copy()

eligible_meta = (
    siginfo_gctx
    .loc[
        (siginfo_gctx["pert_type"] == "trt_cp")
        & (siginfo_gctx["pert_time"] == 24.0)
        & (siginfo_gctx["pert_time_unit"] == "h")
        & (siginfo_gctx["cell_iname"].isin(TARGET_CELLS))
        & (siginfo_gctx["nsample"] >= 3)
    ]
    .copy()
)

compound_annotation = (
    compoundinfo[["pert_id", "target", "moa", "compound_aliases"]]
    .assign(pert_id=lambda x: x["pert_id"].astype(str))
    .drop_duplicates("pert_id")
)

eligible_meta = eligible_meta.merge(compound_annotation, on="pert_id", how="left")

eligible_meta["is_vitd_core_signature"] = eligible_meta["sig_id"].isin(vitd_core_sig_ids)
eligible_meta["is_vitd_candidate_signature"] = eligible_meta["sig_id"].isin(vitd_all_sig_ids)

TEXT_COLUMNS_FOR_CLASSIFICATION = ["cmap_name", "target", "moa", "compound_aliases"]

annotation_text = (
    eligible_meta[TEXT_COLUMNS_FOR_CLASSIFICATION]
    .fillna("")
    .astype(str)
    .agg(" | ".join, axis=1)
    .str.lower()
)

VITD_REGEX = (
    r"vitamin\s*d|vitamin-d|\bvdr\b|calcitriol|calcipotriol|maxacalcitol|"
    r"seocalcitol|ercalcitriol|tacalcitol|paricalcitol|calciferol|"
    r"cholecalciferol|ergocalciferol"
)

NUCLEAR_RECEPTOR_REGEX = (
    r"nuclear receptor|estrogen receptor|\besr1\b|\besr2\b|\besrra\b|\besrrb\b|\besrrg\b|"
    r"\bestradiol\b|\bestrogen\b|androgen receptor|\bar\b|progesterone receptor|\bpgr\b|"
    r"glucocorticoid receptor|\bnr3c1\b|mineralocorticoid receptor|\bnr3c2\b|"
    r"retinoic acid receptor|\brara\b|\brarb\b|\brarg\b|retinoid x receptor|"
    r"\brxra\b|\brxrb\b|\brxrg\b|\bppara\b|\bppard\b|\bpparg\b|ppar|"
    r"thyroid hormone receptor|\bthra\b|\bthrb\b|liver x receptor|\blxr\b|"
    r"\bnr1h2\b|\bnr1h3\b|farnesoid x receptor|\bfxr\b|\bnr1h4\b|"
    r"pregnane x receptor|\bpxr\b|\bnr1i2\b|constitutive androstane receptor|"
    r"\bnr1i3\b|\brora\b|\brorb\b|\brorc\b|\bnr1d1\b|\bnr1d2\b"
)

eligible_meta["is_vitd_annotation"] = annotation_text.str.contains(VITD_REGEX, regex=True, na=False)
eligible_meta["is_nuclear_receptor_annotation"] = annotation_text.str.contains(NUCLEAR_RECEPTOR_REGEX, regex=True, na=False)

eligible_meta["perturbation_group"] = "unrelated_comparator"

eligible_meta.loc[
    eligible_meta["is_nuclear_receptor_annotation"] & ~eligible_meta["is_vitd_annotation"],
    "perturbation_group",
] = "nuclear_receptor_comparator"

eligible_meta.loc[
    eligible_meta["is_vitd_core_signature"],
    "perturbation_group",
] = "vitamin_D_related"

eligible_meta["exclude_from_comparator"] = eligible_meta["is_vitd_annotation"] & ~eligible_meta["is_vitd_core_signature"]

analysis_meta = eligible_meta.loc[~eligible_meta["exclude_from_comparator"]].copy()

if analysis_meta["sig_id"].duplicated().any():
    raise ValueError("Duplicated sig_id values detected in analysis_meta.")

if analysis_meta["is_vitd_core_signature"].sum() != len(vitd_core_sig_ids):
    missing_vitd = sorted(vitd_core_sig_ids - set(analysis_meta["sig_id"]))
    raise ValueError(f"Missing manuscript vitamin D signatures: {missing_vitd[:10]}")

print("Analysis groups")
print("-" * 80)
display(analysis_meta["perturbation_group"].value_counts().rename("n_signatures").to_frame())

print("\nAnalysis groups by cell line")
print("-" * 80)
display(pd.crosstab(analysis_meta["cell_iname"], analysis_meta["perturbation_group"]).reindex(TARGET_CELLS))

print("\nUnique compounds by group")
print("-" * 80)
display(
    analysis_meta
    .groupby("perturbation_group")["pert_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_unique_compounds")
    .to_frame()
)

print("\nExcluded vitamin D/VDR-like non-manuscript signatures:", int(eligible_meta["exclude_from_comparator"].sum()))


## 3. Define fixed core and selected Hallmark programs

The canonical vitamin D core is loaded from `core_v2.pkl`, corresponding to 42 UP and 35 DOWN genes.

The project score definition is:

\[
core\ score = mean(z(core\ UP)) - mean(z(core\ DOWN))
\]

where \(z = (x - mean(x_{all\ genes})) / sd(x_{all\ genes})\) within each signature.

For the fixed-core score, the per-signature mean cancels in the UP-DOWN subtraction:

\[
core\ score =
\frac{mean(core\ UP_{raw}) - mean(core\ DOWN_{raw})}{sd(x_{all\ genes})}
\]

This identity allows efficient block-wise scoring without retaining the full GCTX matrix in memory.


In [ ]:
with open(CANONICAL_CORE_PATH, "rb") as f:
    core_up_ids, core_down_ids = pickle.load(f)

core_up_ids = set(map(str, core_up_ids))
core_down_ids = set(map(str, core_down_ids))
core_gene_ids = sorted(core_up_ids | core_down_ids)

if core_up_ids & core_down_ids:
    raise ValueError("Core UP and DOWN gene sets overlap.")

missing_core_genes = sorted(set(core_gene_ids) - gctx_row_id_set)
if missing_core_genes:
    raise ValueError(f"Core genes missing from GCTX: {missing_core_genes[:10]}")

canonical_top50_reference = (
    pd.read_csv(CORE_SCORES_TOP50_PATH)[["sig_id", "core_score"]]
    .assign(sig_id=lambda x: x["sig_id"].astype(str))
    .set_index("sig_id")["core_score"]
)

SELECTED_HALLMARKS = [
    "HALLMARK_UNFOLDED_PROTEIN_RESPONSE",
    "HALLMARK_HYPOXIA",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_XENOBIOTIC_METABOLISM",
    "HALLMARK_GLYCOLYSIS",
    "HALLMARK_MTORC1_SIGNALING",
    "HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY",
    "HALLMARK_UV_RESPONSE_UP",
    "HALLMARK_UV_RESPONSE_DN",
    "HALLMARK_KRAS_SIGNALING_UP",
    "HALLMARK_APOPTOSIS",
]


def read_gmt(path: Path) -> dict[str, list[str]]:
    gene_sets = {}

    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                gene_sets[parts[0]] = sorted(set(parts[2:]))

    return gene_sets


hallmark_gene_sets_symbols = read_gmt(HALLMARK_GMT_PATH)

missing_hallmarks = sorted(set(SELECTED_HALLMARKS) - set(hallmark_gene_sets_symbols))
if missing_hallmarks:
    raise ValueError(f"Selected Hallmarks not found in GMT: {missing_hallmarks}")

gene_symbol_to_ids = (
    geneinfo
    .assign(
        gene_id=lambda x: x["gene_id"].astype(str),
        gene_symbol=lambda x: x["gene_symbol"].astype(str),
    )
    .dropna(subset=["gene_symbol", "gene_id"])
    .groupby("gene_symbol")["gene_id"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

selected_hallmark_gene_sets = {}

for hallmark in SELECTED_HALLMARKS:
    mapped_gene_ids = []

    for symbol in hallmark_gene_sets_symbols[hallmark]:
        mapped_gene_ids.extend(gene_symbol_to_ids.get(symbol, []))

    selected_hallmark_gene_sets[hallmark] = sorted(set(mapped_gene_ids) & gctx_row_id_set)

hallmark_overlap_summary = pd.DataFrame(
    [
        {
            "hallmark": hallmark,
            "n_symbols_in_gmt": len(hallmark_gene_sets_symbols[hallmark]),
            "n_lincs_mapped_genes": len(selected_hallmark_gene_sets[hallmark]),
            "mapped_fraction": len(selected_hallmark_gene_sets[hallmark]) / len(hallmark_gene_sets_symbols[hallmark]),
        }
        for hallmark in SELECTED_HALLMARKS
    ]
)

low_overlap_hallmarks = hallmark_overlap_summary.loc[
    hallmark_overlap_summary["n_lincs_mapped_genes"] < 10,
    "hallmark",
].tolist()

if low_overlap_hallmarks:
    raise ValueError(f"Some selected Hallmarks have too few mapped LINCS genes: {low_overlap_hallmarks}")

print("Canonical fixed core")
print("-" * 80)
print("UP genes:", len(core_up_ids))
print("DOWN genes:", len(core_down_ids))
print("Total core genes:", len(core_gene_ids))

print("\nSelected Hallmark overlap summary")
print("-" * 80)
display(hallmark_overlap_summary.sort_values("n_lincs_mapped_genes", ascending=False))


## 4. Score fixed core and Hallmark programs

This section performs a single block-wise pass through the Level 5 GCTX matrix. For each signature, it computes the full-signature mean and standard deviation, the fixed vitamin D core score, and selected Hallmark scores.

The full matrix is not loaded into memory at once.


In [ ]:
analysis_sig_ids = analysis_meta["sig_id"].astype(str).tolist()

gctx_signature_index = {sig_id: i for i, sig_id in enumerate(gctx_col_ids)}
gctx_gene_index = {gene_id: i for i, gene_id in enumerate(gctx_row_ids)}

missing_analysis_sigs = sorted(set(analysis_sig_ids) - set(gctx_signature_index))
if missing_analysis_sigs:
    raise ValueError(f"Analysis signatures missing from GCTX: {missing_analysis_sigs[:10]}")

sig_h5_indices = np.array([gctx_signature_index[sig_id] for sig_id in analysis_sig_ids], dtype=int)
sig_sort_order = np.argsort(sig_h5_indices)

sig_h5_indices_sorted = sig_h5_indices[sig_sort_order]
sig_ids_sorted = np.array(analysis_sig_ids, dtype=object)[sig_sort_order].tolist()

core_up_positions = np.array([gctx_gene_index[gene_id] for gene_id in sorted(core_up_ids)], dtype=int)
core_down_positions = np.array([gctx_gene_index[gene_id] for gene_id in sorted(core_down_ids)], dtype=int)

hallmark_gene_positions = {
    hallmark: np.array([gctx_gene_index[gene_id] for gene_id in selected_hallmark_gene_sets[hallmark]], dtype=int)
    for hallmark in SELECTED_HALLMARKS
}

n_signatures = len(sig_h5_indices_sorted)

signature_mean_values = np.empty(n_signatures, dtype=np.float64)
signature_sd_values = np.empty(n_signatures, dtype=np.float64)
core_raw_difference_values = np.empty(n_signatures, dtype=np.float64)
fixed_core_score_values = np.empty(n_signatures, dtype=np.float64)

hallmark_score_values = {
    hallmark: np.empty(n_signatures, dtype=np.float64)
    for hallmark in SELECTED_HALLMARKS
}

with h5py.File(RAW_FILES["level5"], "r") as h5:
    matrix = h5["0/DATA/0/matrix"]

    if matrix.shape != expected_shape:
        raise ValueError(f"Unexpected GCTX matrix shape: {matrix.shape}; expected {expected_shape}")

    n_blocks = int(np.ceil(n_signatures / BLOCK_SIZE))

    for block_id, start in enumerate(range(0, n_signatures, BLOCK_SIZE), start=1):
        stop = min(start + BLOCK_SIZE, n_signatures)
        block_indices = sig_h5_indices_sorted[start:stop]

        block = np.asarray(matrix[block_indices, :], dtype=np.float64)

        block_mean = np.nanmean(block, axis=1)
        block_sd = np.nanstd(block, axis=1, ddof=0)
        safe_sd = np.where((block_sd == 0) | np.isnan(block_sd), np.nan, block_sd)

        signature_mean_values[start:stop] = block_mean
        signature_sd_values[start:stop] = block_sd

        core_raw_difference = (
            np.nanmean(block[:, core_up_positions], axis=1)
            - np.nanmean(block[:, core_down_positions], axis=1)
        )

        core_raw_difference_values[start:stop] = core_raw_difference
        fixed_core_score_values[start:stop] = np.where(
            np.isnan(safe_sd),
            core_raw_difference,
            core_raw_difference / safe_sd,
        )

        for hallmark in SELECTED_HALLMARKS:
            positions = hallmark_gene_positions[hallmark]
            raw_set_mean = np.nanmean(block[:, positions], axis=1)

            hallmark_score_values[hallmark][start:stop] = np.where(
                np.isnan(safe_sd),
                raw_set_mean - block_mean,
                (raw_set_mean - block_mean) / safe_sd,
            )

        if block_id == 1 or block_id % 20 == 0 or block_id == n_blocks:
            print(f"Processed block {block_id}/{n_blocks} ({stop}/{n_signatures} signatures)")

score_table_sorted = pd.DataFrame(
    {
        "sig_id": sig_ids_sorted,
        "signature_mean_all_genes": signature_mean_values,
        "signature_sd_all_genes": signature_sd_values,
        "core_raw_difference": core_raw_difference_values,
        "fixed_core_score": fixed_core_score_values,
        **{f"{hallmark}_score": hallmark_score_values[hallmark] for hallmark in SELECTED_HALLMARKS},
    }
)

score_table = score_table_sorted.set_index("sig_id").loc[analysis_sig_ids].reset_index()

analysis_meta_pathway_scored = analysis_meta.merge(score_table, on="sig_id", how="left")

hallmark_score_columns = [f"{hallmark}_score" for hallmark in SELECTED_HALLMARKS]
required_score_columns = ["fixed_core_score", *hallmark_score_columns]

if analysis_meta_pathway_scored[required_score_columns].isna().any().any():
    na_counts = analysis_meta_pathway_scored[required_score_columns].isna().sum()
    raise ValueError("Missing score values detected:\n" + na_counts[na_counts > 0].to_string())

vitd_score_check = (
    analysis_meta_pathway_scored
    .loc[analysis_meta_pathway_scored["perturbation_group"].eq("vitamin_D_related"), ["sig_id", "fixed_core_score"]]
    .merge(canonical_top50_reference.rename("archived_top50_core_score").reset_index(), on="sig_id", how="inner")
)

vitd_score_check["abs_diff"] = (
    vitd_score_check["fixed_core_score"]
    - vitd_score_check["archived_top50_core_score"]
).abs()

max_vitd_score_diff = vitd_score_check["abs_diff"].max()

if max_vitd_score_diff > NUMERICAL_TOLERANCE:
    raise ValueError(
        "Projected fixed_core_score does not reproduce archived top50 scores within tolerance. "
        f"Max absolute difference: {max_vitd_score_diff}"
    )

print("\nScoring completed")
print("-" * 80)
print("Scored metadata shape:", analysis_meta_pathway_scored.shape)
print("Vitamin D signatures validated:", vitd_score_check.shape[0])
print("Max absolute difference vs archived top50:", max_vitd_score_diff)

print("\nFixed core score summary by group")
print("-" * 80)
display(
    analysis_meta_pathway_scored
    .groupby("perturbation_group")["fixed_core_score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

print("\nHallmark score mean by group")
print("-" * 80)
display(
    analysis_meta_pathway_scored
    .groupby("perturbation_group")[hallmark_score_columns]
    .mean()
    .T
    .rename_axis("hallmark_score")
    .reset_index()
)


## 5. Fixed-core score specificity

This section compares the fixed vitamin D core score across perturbation groups. Because comparator groups are much larger than the vitamin D group, interpretation emphasizes effect sizes, directionality, and matched resampling rather than p-values alone.


In [ ]:
def cliffs_delta_from_mannwhitney(x, y):
    x = pd.Series(x, dtype=float).dropna()
    y = pd.Series(y, dtype=float).dropna()

    if x.empty or y.empty:
        return np.nan

    u_stat = stats.mannwhitneyu(x, y, alternative="two-sided").statistic
    return float((2 * u_stat) / (len(x) * len(y)) - 1)


def hedges_g(x, y):
    x = pd.Series(x, dtype=float).dropna()
    y = pd.Series(y, dtype=float).dropna()

    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan

    sx = x.std(ddof=1)
    sy = y.std(ddof=1)
    pooled_sd = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))

    if pooled_sd == 0 or np.isnan(pooled_sd):
        return np.nan

    cohen_d = (x.mean() - y.mean()) / pooled_sd
    correction = 1 - (3 / (4 * (nx + ny) - 9))

    return float(cohen_d * correction)


def compare_groups(frame, reference_group, comparator_group, score_col):
    ref = frame.loc[frame["perturbation_group"].eq(reference_group), score_col].dropna()
    comp = frame.loc[frame["perturbation_group"].eq(comparator_group), score_col].dropna()

    welch = stats.ttest_ind(ref, comp, equal_var=False, nan_policy="omit")
    mw = stats.mannwhitneyu(ref, comp, alternative="two-sided")

    return {
        "reference_group": reference_group,
        "comparator_group": comparator_group,
        "n_reference": len(ref),
        "n_comparator": len(comp),
        "mean_reference": ref.mean(),
        "mean_comparator": comp.mean(),
        "mean_difference": ref.mean() - comp.mean(),
        "median_reference": ref.median(),
        "median_comparator": comp.median(),
        "median_difference": ref.median() - comp.median(),
        "welch_t": welch.statistic,
        "welch_p": welch.pvalue,
        "mannwhitney_u": mw.statistic,
        "mannwhitney_p": mw.pvalue,
        "cliffs_delta": cliffs_delta_from_mannwhitney(ref, comp),
        "hedges_g": hedges_g(ref, comp),
    }


CORE_GROUP_COMPARISONS = [
    ("vitamin_D_related", "unrelated_comparator"),
    ("vitamin_D_related", "nuclear_receptor_comparator"),
    ("nuclear_receptor_comparator", "unrelated_comparator"),
]

global_comparisons = pd.DataFrame(
    [
        compare_groups(analysis_meta_pathway_scored, reference_group, comparator_group, "fixed_core_score")
        for reference_group, comparator_group in CORE_GROUP_COMPARISONS
    ]
)

global_comparisons["mannwhitney_fdr"] = multipletests(global_comparisons["mannwhitney_p"], method="fdr_bh")[1]
global_comparisons["welch_fdr"] = multipletests(global_comparisons["welch_p"], method="fdr_bh")[1]

print("Global fixed-core score comparisons")
print("-" * 80)
display(
    global_comparisons[
        [
            "reference_group", "comparator_group", "n_reference", "n_comparator",
            "mean_difference", "median_difference", "cliffs_delta", "hedges_g",
            "mannwhitney_p", "mannwhitney_fdr", "welch_p", "welch_fdr",
        ]
    ]
)

print("\nFixed-core score summary by group and cell line")
print("-" * 80)
display(
    analysis_meta_pathway_scored
    .groupby(["perturbation_group", "cell_iname"])["fixed_core_score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
    .sort_values(["cell_iname", "perturbation_group"])
)


## 6. Cell-line-matched and cell-line-stratified fixed-core analyses

The matched analysis samples comparator signatures to reproduce the vitamin D cell-line composition. The stratified analysis checks whether the direction of the effect is consistent within each cell line.


In [ ]:
vitd_group = analysis_meta_pathway_scored[
    analysis_meta_pathway_scored["perturbation_group"].eq("vitamin_D_related")
].copy()

vitd_cell_counts = vitd_group["cell_iname"].value_counts().reindex(TARGET_CELLS, fill_value=0)

vitd_mean = vitd_group["fixed_core_score"].mean()
vitd_median = vitd_group["fixed_core_score"].median()


def cell_line_matched_resampling(frame, comparator_group, target_cell_counts, n_iterations, score_col, random_seed):
    rng = np.random.default_rng(random_seed)

    comparator = frame[frame["perturbation_group"].eq(comparator_group)].copy()

    comparator_by_cell = {
        cell: comparator.loc[comparator["cell_iname"].eq(cell), score_col].dropna().to_numpy()
        for cell in target_cell_counts.index
    }

    for cell, required_n in target_cell_counts.items():
        if len(comparator_by_cell[cell]) < required_n:
            raise ValueError(
                f"Comparator group '{comparator_group}' has insufficient signatures for {cell}: "
                f"required {required_n}, available {len(comparator_by_cell[cell])}."
            )

    sampled_means = np.empty(n_iterations, dtype=float)
    sampled_medians = np.empty(n_iterations, dtype=float)

    for i in range(n_iterations):
        sampled_values = []

        for cell, required_n in target_cell_counts.items():
            if required_n > 0:
                values = comparator_by_cell[cell]
                sampled_values.append(rng.choice(values, size=required_n, replace=False))

        sampled_values = np.concatenate(sampled_values)
        sampled_means[i] = sampled_values.mean()
        sampled_medians[i] = np.median(sampled_values)

    return pd.DataFrame(
        {
            "comparator_group": comparator_group,
            "iteration": np.arange(n_iterations),
            "sampled_mean": sampled_means,
            "sampled_median": sampled_medians,
        }
    )


matched_resampling = pd.concat(
    [
        cell_line_matched_resampling(
            analysis_meta_pathway_scored,
            comparator_group="unrelated_comparator",
            target_cell_counts=vitd_cell_counts,
            n_iterations=N_CORE_RESAMPLING_ITERATIONS,
            score_col="fixed_core_score",
            random_seed=RANDOM_SEED + 1,
        ),
        cell_line_matched_resampling(
            analysis_meta_pathway_scored,
            comparator_group="nuclear_receptor_comparator",
            target_cell_counts=vitd_cell_counts,
            n_iterations=N_CORE_RESAMPLING_ITERATIONS,
            score_col="fixed_core_score",
            random_seed=RANDOM_SEED + 2,
        ),
    ],
    ignore_index=True,
)

matched_summary_rows = []

for comparator_group, group_df in matched_resampling.groupby("comparator_group"):
    mean_differences = vitd_mean - group_df["sampled_mean"]
    median_differences = vitd_median - group_df["sampled_median"]

    matched_summary_rows.append(
        {
            "reference_group": "vitamin_D_related",
            "comparator_group": comparator_group,
            "n_iterations": group_df.shape[0],
            "matched_n_per_iteration": int(vitd_cell_counts.sum()),
            "vitd_mean": vitd_mean,
            "mean_comparator_matched_mean": group_df["sampled_mean"].mean(),
            "mean_difference": mean_differences.mean(),
            "mean_difference_ci025": mean_differences.quantile(0.025),
            "mean_difference_ci975": mean_differences.quantile(0.975),
            "empirical_p_mean_one_sided": ((mean_differences <= 0).sum() + 1) / (len(mean_differences) + 1),
            "vitd_median": vitd_median,
            "mean_comparator_matched_median": group_df["sampled_median"].mean(),
            "median_difference": median_differences.mean(),
            "median_difference_ci025": median_differences.quantile(0.025),
            "median_difference_ci975": median_differences.quantile(0.975),
            "empirical_p_median_one_sided": ((median_differences <= 0).sum() + 1) / (len(median_differences) + 1),
        }
    )

matched_resampling_summary = pd.DataFrame(matched_summary_rows)

cell_line_comparison_rows = []

for cell in TARGET_CELLS:
    cell_df = analysis_meta_pathway_scored[analysis_meta_pathway_scored["cell_iname"].eq(cell)].copy()

    for comparator_group in ["unrelated_comparator", "nuclear_receptor_comparator"]:
        ref = cell_df.loc[cell_df["perturbation_group"].eq("vitamin_D_related"), "fixed_core_score"].dropna()
        comp = cell_df.loc[cell_df["perturbation_group"].eq(comparator_group), "fixed_core_score"].dropna()

        welch = stats.ttest_ind(ref, comp, equal_var=False, nan_policy="omit")
        mw = stats.mannwhitneyu(ref, comp, alternative="two-sided")

        cell_line_comparison_rows.append(
            {
                "cell_iname": cell,
                "reference_group": "vitamin_D_related",
                "comparator_group": comparator_group,
                "n_reference": len(ref),
                "n_comparator": len(comp),
                "mean_reference": ref.mean(),
                "mean_comparator": comp.mean(),
                "mean_difference": ref.mean() - comp.mean(),
                "median_reference": ref.median(),
                "median_comparator": comp.median(),
                "median_difference": ref.median() - comp.median(),
                "cliffs_delta": cliffs_delta_from_mannwhitney(ref, comp),
                "hedges_g": hedges_g(ref, comp),
                "mannwhitney_p": mw.pvalue,
                "welch_p": welch.pvalue,
            }
        )

cell_line_comparisons = pd.DataFrame(cell_line_comparison_rows)
cell_line_comparisons["mannwhitney_fdr"] = multipletests(cell_line_comparisons["mannwhitney_p"], method="fdr_bh")[1]
cell_line_comparisons["welch_fdr"] = multipletests(cell_line_comparisons["welch_p"], method="fdr_bh")[1]

directionality_summary = (
    cell_line_comparisons
    .assign(
        positive_mean_difference=lambda x: x["mean_difference"] > 0,
        positive_median_difference=lambda x: x["median_difference"] > 0,
    )
    .groupby("comparator_group")
    .agg(
        n_cell_lines=("cell_iname", "count"),
        n_positive_mean_difference=("positive_mean_difference", "sum"),
        n_positive_median_difference=("positive_median_difference", "sum"),
        min_mean_difference=("mean_difference", "min"),
        max_mean_difference=("mean_difference", "max"),
        min_median_difference=("median_difference", "min"),
        max_median_difference=("median_difference", "max"),
    )
    .reset_index()
)

print("Vitamin D cell-line composition used for matching")
print("-" * 80)
display(vitd_cell_counts.rename("n_vitamin_D_signatures").to_frame())

print("\nCell-line-matched fixed-core resampling summary")
print("-" * 80)
display(matched_resampling_summary)

print("\nCell-line-stratified fixed-core score comparisons")
print("-" * 80)
display(
    cell_line_comparisons[
        [
            "cell_iname", "reference_group", "comparator_group", "n_reference", "n_comparator",
            "mean_difference", "median_difference", "cliffs_delta", "hedges_g",
            "mannwhitney_p", "mannwhitney_fdr", "welch_p", "welch_fdr",
        ]
    ].sort_values(["cell_iname", "comparator_group"])
)

print("\nDirectionality summary")
print("-" * 80)
display(directionality_summary)


## 7. Hallmark pathway specificity

Selected Hallmark scores are compared globally and using the same cell-line-matched resampling strategy.


In [ ]:
PATHWAY_COMPARISONS = [
    ("vitamin_D_related", "unrelated_comparator"),
    ("vitamin_D_related", "nuclear_receptor_comparator"),
    ("nuclear_receptor_comparator", "unrelated_comparator"),
]

pathway_global_comparison_rows = []

for hallmark_score_col in hallmark_score_columns:
    hallmark = hallmark_score_col.replace("_score", "")

    for reference_group, comparator_group in PATHWAY_COMPARISONS:
        pathway_global_comparison_rows.append(
            {
                "hallmark": hallmark,
                **compare_groups(
                    analysis_meta_pathway_scored,
                    reference_group=reference_group,
                    comparator_group=comparator_group,
                    score_col=hallmark_score_col,
                ),
            }
        )

pathway_global_comparisons = pd.DataFrame(pathway_global_comparison_rows)
pathway_global_comparisons["mannwhitney_fdr"] = multipletests(pathway_global_comparisons["mannwhitney_p"], method="fdr_bh")[1]
pathway_global_comparisons["welch_fdr"] = multipletests(pathway_global_comparisons["welch_p"], method="fdr_bh")[1]
pathway_global_comparisons["vitamin_d_vs_comparator"] = pathway_global_comparisons["reference_group"].eq("vitamin_D_related")

print("Global Hallmark comparisons")
print("-" * 80)
display(
    pathway_global_comparisons[
        [
            "hallmark", "reference_group", "comparator_group", "n_reference", "n_comparator",
            "mean_difference", "median_difference", "cliffs_delta", "hedges_g",
            "mannwhitney_p", "mannwhitney_fdr", "welch_p", "welch_fdr",
        ]
    ].sort_values(["reference_group", "comparator_group", "mean_difference"], ascending=[True, True, False])
)


In [ ]:
vitd_pathway_group = analysis_meta_pathway_scored[
    analysis_meta_pathway_scored["perturbation_group"].eq("vitamin_D_related")
].copy()

vitd_pathway_means = vitd_pathway_group[hallmark_score_columns].mean(axis=0)
vitd_pathway_medians = vitd_pathway_group[hallmark_score_columns].median(axis=0)


def pathway_cell_line_matched_resampling(frame, comparator_group, target_cell_counts, score_columns, n_iterations, random_seed):
    rng = np.random.default_rng(random_seed)
    comparator = frame[frame["perturbation_group"].eq(comparator_group)].copy()

    comparator_by_cell = {
        cell: comparator.loc[comparator["cell_iname"].eq(cell), score_columns].dropna().to_numpy()
        for cell in target_cell_counts.index
    }

    for cell, required_n in target_cell_counts.items():
        available_n = comparator_by_cell[cell].shape[0]
        if available_n < required_n:
            raise ValueError(
                f"Comparator group '{comparator_group}' has insufficient signatures for {cell}: "
                f"required {required_n}, available {available_n}."
            )

    sampled_means = np.empty((n_iterations, len(score_columns)), dtype=float)
    sampled_medians = np.empty((n_iterations, len(score_columns)), dtype=float)

    for i in range(n_iterations):
        sampled_blocks = []

        for cell, required_n in target_cell_counts.items():
            if required_n > 0:
                values = comparator_by_cell[cell]
                sampled_idx = rng.choice(values.shape[0], size=required_n, replace=False)
                sampled_blocks.append(values[sampled_idx, :])

        sampled_values = np.vstack(sampled_blocks)

        sampled_means[i, :] = sampled_values.mean(axis=0)
        sampled_medians[i, :] = np.median(sampled_values, axis=0)

    mean_df = pd.DataFrame(sampled_means, columns=score_columns)
    median_df = pd.DataFrame(sampled_medians, columns=score_columns)

    mean_df["iteration"] = np.arange(n_iterations)
    mean_df["comparator_group"] = comparator_group
    mean_df["statistic"] = "mean"

    median_df["iteration"] = np.arange(n_iterations)
    median_df["comparator_group"] = comparator_group
    median_df["statistic"] = "median"

    return pd.concat([mean_df, median_df], ignore_index=True)


pathway_matched_resampling = pd.concat(
    [
        pathway_cell_line_matched_resampling(
            analysis_meta_pathway_scored,
            comparator_group="unrelated_comparator",
            target_cell_counts=vitd_cell_counts,
            score_columns=hallmark_score_columns,
            n_iterations=N_PATHWAY_RESAMPLING_ITERATIONS,
            random_seed=RANDOM_SEED + 11,
        ),
        pathway_cell_line_matched_resampling(
            analysis_meta_pathway_scored,
            comparator_group="nuclear_receptor_comparator",
            target_cell_counts=vitd_cell_counts,
            score_columns=hallmark_score_columns,
            n_iterations=N_PATHWAY_RESAMPLING_ITERATIONS,
            random_seed=RANDOM_SEED + 12,
        ),
    ],
    ignore_index=True,
)

pathway_matched_summary_rows = []

for comparator_group in ["unrelated_comparator", "nuclear_receptor_comparator"]:
    for hallmark_score_col in hallmark_score_columns:
        hallmark = hallmark_score_col.replace("_score", "")

        sampled_mean_values = pathway_matched_resampling.loc[
            (pathway_matched_resampling["comparator_group"].eq(comparator_group))
            & (pathway_matched_resampling["statistic"].eq("mean")),
            hallmark_score_col,
        ].dropna()

        sampled_median_values = pathway_matched_resampling.loc[
            (pathway_matched_resampling["comparator_group"].eq(comparator_group))
            & (pathway_matched_resampling["statistic"].eq("median")),
            hallmark_score_col,
        ].dropna()

        mean_differences = vitd_pathway_means[hallmark_score_col] - sampled_mean_values
        median_differences = vitd_pathway_medians[hallmark_score_col] - sampled_median_values

        pathway_matched_summary_rows.append(
            {
                "hallmark": hallmark,
                "reference_group": "vitamin_D_related",
                "comparator_group": comparator_group,
                "n_iterations": N_PATHWAY_RESAMPLING_ITERATIONS,
                "matched_n_per_iteration": int(vitd_cell_counts.sum()),
                "vitd_mean": vitd_pathway_means[hallmark_score_col],
                "mean_comparator_matched_mean": sampled_mean_values.mean(),
                "mean_difference": mean_differences.mean(),
                "mean_difference_ci025": mean_differences.quantile(0.025),
                "mean_difference_ci975": mean_differences.quantile(0.975),
                "empirical_p_mean_one_sided": ((mean_differences <= 0).sum() + 1) / (len(mean_differences) + 1),
                "vitd_median": vitd_pathway_medians[hallmark_score_col],
                "mean_comparator_matched_median": sampled_median_values.mean(),
                "median_difference": median_differences.mean(),
                "median_difference_ci025": median_differences.quantile(0.025),
                "median_difference_ci975": median_differences.quantile(0.975),
                "empirical_p_median_one_sided": ((median_differences <= 0).sum() + 1) / (len(median_differences) + 1),
            }
        )

pathway_matched_resampling_summary = pd.DataFrame(pathway_matched_summary_rows)

pathway_matched_resampling_summary["empirical_p_mean_fdr"] = multipletests(
    pathway_matched_resampling_summary["empirical_p_mean_one_sided"],
    method="fdr_bh",
)[1]

pathway_matched_resampling_summary["empirical_p_median_fdr"] = multipletests(
    pathway_matched_resampling_summary["empirical_p_median_one_sided"],
    method="fdr_bh",
)[1]

pathway_matched_resampling_summary["vitamin_d_higher_by_mean"] = pathway_matched_resampling_summary["mean_difference"] > 0
pathway_matched_resampling_summary["vitamin_d_higher_by_median"] = pathway_matched_resampling_summary["median_difference"] > 0

pathway_matched_directionality = (
    pathway_matched_resampling_summary
    .groupby("hallmark")
    .agg(
        n_comparisons=("comparator_group", "count"),
        n_higher_by_mean=("vitamin_d_higher_by_mean", "sum"),
        n_higher_by_median=("vitamin_d_higher_by_median", "sum"),
        n_mean_fdr_significant=("empirical_p_mean_fdr", lambda x: (x < 0.05).sum()),
        n_median_fdr_significant=("empirical_p_median_fdr", lambda x: (x < 0.05).sum()),
        min_mean_difference=("mean_difference", "min"),
        max_mean_difference=("mean_difference", "max"),
        min_median_difference=("median_difference", "min"),
        max_median_difference=("median_difference", "max"),
    )
    .reset_index()
    .assign(
        vitamin_d_higher_than_both_by_mean=lambda x: x["n_higher_by_mean"] == 2,
        vitamin_d_higher_than_both_by_median=lambda x: x["n_higher_by_median"] == 2,
        vitamin_d_mean_fdr_significant_vs_both=lambda x: x["n_mean_fdr_significant"] == 2,
        vitamin_d_median_fdr_significant_vs_both=lambda x: x["n_median_fdr_significant"] == 2,
    )
    .sort_values(["vitamin_d_higher_than_both_by_mean", "min_mean_difference"], ascending=[False, False])
)

print("Cell-line-matched Hallmark resampling summary")
print("-" * 80)
display(
    pathway_matched_resampling_summary[
        [
            "hallmark", "comparator_group", "n_iterations",
            "mean_difference", "mean_difference_ci025", "mean_difference_ci975",
            "empirical_p_mean_one_sided", "empirical_p_mean_fdr",
            "median_difference", "median_difference_ci025", "median_difference_ci975",
            "empirical_p_median_one_sided", "empirical_p_median_fdr",
        ]
    ].sort_values(["comparator_group", "mean_difference"], ascending=[True, False])
)

print("\nMatched pathway directionality summary")
print("-" * 80)
display(pathway_matched_directionality)


## 8. Export revision summary table

The exported table consolidates fixed-core score comparisons, cell-line-matched analyses, cell-line-stratified directionality, and Hallmark pathway comparisons.


In [ ]:
COMPARATOR_SPECIFICITY_SUMMARY_PATH = TABLES_DIR / "comparator_specificity_summary.csv"

core_global_export = global_comparisons.copy().assign(
    analysis_family="fixed_core_score",
    analysis_type="global_unmatched",
    feature="fixed_core_score",
)

core_matched_export = matched_resampling_summary.copy().assign(
    analysis_family="fixed_core_score",
    analysis_type="cell_line_matched_resampling",
    feature="fixed_core_score",
)

core_cellline_direction_export = directionality_summary.copy().assign(
    analysis_family="fixed_core_score",
    analysis_type="cell_line_stratified_directionality",
    feature="fixed_core_score",
    reference_group="vitamin_D_related",
)

pathway_global_export = (
    pathway_global_comparisons
    .loc[pathway_global_comparisons["reference_group"].eq("vitamin_D_related")]
    .copy()
    .assign(
        analysis_family="hallmark_score",
        analysis_type="global_unmatched",
        feature=lambda x: x["hallmark"],
    )
)

pathway_matched_export = pathway_matched_resampling_summary.copy().assign(
    analysis_family="hallmark_score",
    analysis_type="cell_line_matched_resampling",
    feature=lambda x: x["hallmark"],
)

pathway_direction_export = pathway_matched_directionality.copy().assign(
    analysis_family="hallmark_score",
    analysis_type="cell_line_matched_directionality",
    feature=lambda x: x["hallmark"],
    reference_group="vitamin_D_related",
    comparator_group="both_comparator_classes",
)

comparator_specificity_summary = pd.concat(
    [
        core_global_export,
        core_matched_export,
        core_cellline_direction_export,
        pathway_global_export,
        pathway_matched_export,
        pathway_direction_export,
    ],
    ignore_index=True,
    sort=False,
)

preferred_column_order = [
    "analysis_family", "analysis_type", "feature", "hallmark", "reference_group", "comparator_group",
    "n_reference", "n_comparator", "n_iterations", "matched_n_per_iteration",
    "mean_reference", "mean_comparator", "mean_difference", "mean_difference_ci025", "mean_difference_ci975",
    "median_reference", "median_comparator", "median_difference", "median_difference_ci025", "median_difference_ci975",
    "cliffs_delta", "hedges_g", "mannwhitney_p", "mannwhitney_fdr", "welch_p", "welch_fdr",
    "empirical_p_mean_one_sided", "empirical_p_mean_fdr",
    "empirical_p_median_one_sided", "empirical_p_median_fdr",
    "n_cell_lines", "n_positive_mean_difference", "n_positive_median_difference",
    "n_comparisons", "n_higher_by_mean", "n_higher_by_median",
    "n_mean_fdr_significant", "n_median_fdr_significant",
    "vitamin_d_higher_than_both_by_mean", "vitamin_d_higher_than_both_by_median",
    "vitamin_d_mean_fdr_significant_vs_both", "vitamin_d_median_fdr_significant_vs_both",
]

existing_preferred_columns = [col for col in preferred_column_order if col in comparator_specificity_summary.columns]
remaining_columns = [col for col in comparator_specificity_summary.columns if col not in existing_preferred_columns]

comparator_specificity_summary = comparator_specificity_summary[existing_preferred_columns + remaining_columns]
comparator_specificity_summary.to_csv(COMPARATOR_SPECIFICITY_SUMMARY_PATH, index=False)

robust_hallmarks_vs_both = (
    pathway_matched_directionality
    .loc[
        pathway_matched_directionality["vitamin_d_mean_fdr_significant_vs_both"]
        & pathway_matched_directionality["vitamin_d_median_fdr_significant_vs_both"],
        "hallmark",
    ]
    .tolist()
)

partial_or_nonrobust_hallmarks = (
    pathway_matched_directionality
    .loc[
        ~(
            pathway_matched_directionality["vitamin_d_mean_fdr_significant_vs_both"]
            & pathway_matched_directionality["vitamin_d_median_fdr_significant_vs_both"]
        ),
        "hallmark",
    ]
    .tolist()
)

print("Saved comparator-specificity summary")
print("-" * 80)
print(COMPARATOR_SPECIFICITY_SUMMARY_PATH.relative_to(PROJECT_ROOT))
print("Rows:", comparator_specificity_summary.shape[0])
print("Columns:", comparator_specificity_summary.shape[1])

print("\nCore score: cell-line-matched vitamin D differences")
print("-" * 80)
display(
    matched_resampling_summary[
        [
            "comparator_group",
            "mean_difference", "mean_difference_ci025", "mean_difference_ci975", "empirical_p_mean_one_sided",
            "median_difference", "median_difference_ci025", "median_difference_ci975", "empirical_p_median_one_sided",
        ]
    ]
)

print("\nHallmarks robustly higher in vitamin D than both comparator classes")
print("-" * 80)
print(f"{len(robust_hallmarks_vs_both)} / {len(SELECTED_HALLMARKS)}")
for hallmark in robust_hallmarks_vs_both:
    print("-", hallmark)

print("\nPartial or non-robust Hallmarks")
print("-" * 80)
for hallmark in partial_or_nonrobust_hallmarks:
    print("-", hallmark)


## Interpretation

This comparator analysis was designed to address whether the vitamin D transcriptional core and recurrent pathway signals merely reflect generic perturbational responses.

A fixed vitamin D core was projected onto the original vitamin D-related signatures, non-VDR nuclear receptor comparator signatures, and unrelated compound perturbations. The core was not re-derived in comparator signatures, thereby avoiding circularity.

The fixed vitamin D core score was markedly higher in vitamin D-related signatures than in both comparator classes. This result remained robust after matching comparator samples to the vitamin D cell-line composition. In matched resampling, vitamin D signatures exceeded nuclear receptor comparators by approximately 1.34 fixed-core score units and unrelated comparators by approximately 1.42 fixed-core score units, with empirical one-sided p = 0.0002 in both comparisons.

The effect was also consistent across all five manuscript cell lines. Within each cell line, vitamin D signatures showed higher fixed-core scores than both comparator classes, indicating that the result was not driven by a single cellular context.

At the pathway level, selected Hallmark scores were evaluated using full-signature standardization. Eight of twelve tested Hallmarks were robustly higher in vitamin D signatures than in both comparator groups after cell-line-matched resampling: hypoxia, TNFA/NF-kB signaling, UV response down, glycolysis, reactive oxygen species pathway, inflammatory response, xenobiotic metabolism, and KRAS signaling up. In contrast, unfolded protein response was not robustly higher than both comparator classes, indicating that not all stress-associated pathways behave as vitamin D-enriched signals.

Overall, these results support a revised interpretation: the vitamin D transcriptional response includes pathway programs that can also occur in generic perturbational contexts, but the fixed core score and several pathway-level signals are quantitatively enriched in vitamin D-related signatures relative to both unrelated compounds and non-VDR nuclear receptor perturbations. These findings support biological specificity at the level of relative enrichment rather than absolute exclusivity.
